# ANÁLISIS DE MARKETING
Semana: 2
¿Cuál es el impacto del tipo de contacto, ya sea móvil o telefónico, a la tasa de conversión de nuestras campañas de marketing?

_____

In [1]:
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
import numpy as np

# Datos y DF

In [ ]:
# Cargar datos ### RUTA LOCAL ### 
df_marketing = pd.read_csv(".../Data/06-01-2026/06-01-2026_Clean.csv")

In [17]:
df_marketing["perfil_deuda"] = np.where(
    (df_marketing["housing"] == 1) | (df_marketing["loan"] == 1), 1, 0
)

In [18]:
display(df_marketing)

,id,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit,perfil_deuda
0,1,59,admin.,married,secondary,0,2343,1,0,unknown,5,may,1042,1,-1,0,no_campaign,1,1
1,2,56,admin.,married,secondary,0,45,0,0,unknown,5,may,1467,1,-1,0,no_campaign,1,0
2,3,41,technician,married,secondary,0,1270,1,0,unknown,5,may,1389,1,-1,0,no_campaign,1,1
3,4,55,services,married,secondary,0,2476,1,0,unknown,5,may,579,1,-1,0,no_campaign,1,1
4,5,54,admin.,married,tertiary,0,184,0,0,unknown,5,may,673,2,-1,0,no_campaign,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10982,10983,40,management,married,secondary,0,8486,0,0,unknown,6,may,260,3,-1,0,no_campaign,0,0
10983,10984,53,management,married,tertiary,0,20772,0,0,cellular,4,feb,715,1,-1,0,no_campaign,0,0
10984,10985,55,blue-collar,married,primary,0,3297,1,1,telephone,30,apr,96,1,-1,0,no_campaign,0,1
10985,10986,41,management,married,tertiary,0,9,1,0,cellular,22,jul,82,3,-1,0,no_campaign,0,1


In [19]:
df_marketing.dtypes

id               int64
age              int64
job             object
marital         object
education       object
default          int64
balance          int64
housing          int64
loan             int64
contact         object
day              int64
month           object
duration         int64
campaign         int64
pdays            int64
previous         int64
poutcome        object
deposit          int64
perfil_deuda     int64
dtype: object

In [21]:
# 1. Definir los productos y los canales
productos = ['housing', 'loan', 'deposit']
canales = ['telephone', 'cellular', 'unknown']

filas = []

# 2. Calcular datos para cada producto
for prod in productos:
    # Filtrar solo los clientes que sí tienen el producto
    df_prod = df_marketing[df_marketing[prod] == 1]
    
    for canal in canales:
        # Filtrar por canal de contacto
        df_canal = df_prod[df_prod['contact'] == canal]
        
        # Contar clientes y sumar el total de contactos 
        num_clientes = len(df_canal)
        total_contactos = df_canal['campaign'].sum()
        
        filas.append({
            'Producto / Estado': prod.capitalize(),
            'Canal de Contacto': canal.capitalize(),
            'Cantidad de Clientes': num_clientes,
            'Total de Contactos': total_contactos
        })

# 3. Calcular datos para los que NO tienen ningún producto
df_ninguno = df_marketing[(df_marketing['housing'] == 0) & 
                          (df_marketing['loan'] == 0) & 
                          (df_marketing['deposit'] == 0)]

for canal in canales:
    df_canal = df_ninguno[df_ninguno['contact'] == canal]
    num_clientes = len(df_canal)
    total_contactos = df_canal['campaign'].sum()
    
    filas.append({
        'Producto / Estado': 'Ninguno de los tres',
        'Canal de Contacto': canal.capitalize(),
        'Cantidad de Clientes': num_clientes,
        'Total de Contactos': total_contactos
    })

# 4. Crear DataFrame 
tabla_final = pd.DataFrame(filas)
tabla_final.set_index(['Producto / Estado', 'Canal de Contacto'], inplace=True)

display(tabla_final)


Cantidad de Clientes  \
Producto / Estado   Canal de Contacto                         
Housing             Telephone                           239   
                    Cellular                           3291   
                    Unknown                            1653   
Loan                Telephone                            80   
                    Cellular                           1031   
                    Unknown                             320   
Deposit             Telephone                           390   
                    Cellular                           4369   
                    Unknown                             530   
Ninguno de los tres Telephone                           182   
                    Cellular                           1454   
                    Unknown                             420   

                                       Total de Contactos  
Producto / Estado   Canal de Contacto                      
Housing             Telephone                         888  
                    Cellular                         7556  
                    Unknown                          4662  
Loan                Telephone                         330  
                    Cellular                         2670  
                    Unknown                           939  
Deposit             Telephone                         935  
                    Cellular                         9077  
                    Unknown                          1312  
Ninguno de los tres Telephone                         539  
                    Cellular                         4432  
                    Unknown                          1157

In [23]:
# Revisar el cruce de las variables principales para asegurar volumen de datos
tabla_cruce = pd.crosstab(df_marketing['contact'], df_marketing['deposit'])
print("--- Validación de Tamaño de Muestra ---")
print(tabla_cruce)


--- Validación de Tamaño de Muestra ---
deposit       0     1
contact              
cellular   3559  4369
telephone   374   390
unknown    1765   530


_____
# Modelo de regresión logística

MODELO BASE

In [24]:
# Ajustar el modelo incluyendo la variable de interés y las variables de control seleccionadas
formula = "deposit ~ C(contact) + age + C(perfil_deuda) + C(poutcome) + campaign"
modelo = smf.logit(formula, data=df_marketing).fit()

# Ver el resumen estadístico completo
print(modelo.summary())

Optimization terminated successfully.
         Current function value: 0.602251
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:                deposit   No. Observations:                10987
Model:                          Logit   Df Residuals:                    10978
Method:                           MLE   Df Model:                            8
Date:                Thu, 04 Jun 2026   Pseudo R-squ.:                  0.1303
Time:                        16:55:15   Log-Likelihood:                -6616.9
converged:                       True   LL-Null:                       -7608.0
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                 coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------
Intercept                      0.7835      0.104      7.568      0.000      

In [25]:
# Extraer los Odds Ratios y sus P-valores para medir la significancia estadística
resultados = pd.DataFrame({
    'Odds Ratio (OR)': np.exp(modelo.params),
    'P-valor': modelo.pvalues
})

print("\n--- RESULTADOS FINALES DEL MODELO ---")
print(resultados.round(4))



--- RESULTADOS FINALES DEL MODELO ---
                            Odds Ratio (OR)  P-valor
Intercept                            2.1892   0.0000
C(contact)[T.telephone]              0.8514   0.0531
C(contact)[T.unknown]                0.3696   0.0000
C(perfil_deuda)[T.1]                 0.4869   0.0000
C(poutcome)[T.no_campaign]           0.8636   0.0240
C(poutcome)[T.other]                 1.2933   0.0166
C(poutcome)[T.success]               8.4832   0.0000
age                                  0.9979   0.2559
campaign                             0.9044   0.0000


____
# MODELO BASE + BALANCE

In [34]:
# Crear variable de tramo de balance ## 127€ saldo bajo segun perfil en FINANZAS 
condiciones = [
    (df_marketing['balance'] <= 0),
    (df_marketing['balance'] > 0) & (df_marketing['balance'] <= 127), ## 127 segun perfil finanzas
    (df_marketing['balance'] > 127) & (df_marketing['balance'] <= 2000),
    (df_marketing['balance'] > 2000)
]
opciones = ['1. Negativo o Cero', '2. Saldo Bajo', '3. Saldo Medio', '4. Saldo Alto']
df_marketing['tramo_balance'] = np.select(condiciones, opciones, default='2. Saldo Bajo')

In [35]:
# Ajustar el modelo incluyendo las interacciones y el nuevo tramo de balance
# contact * age = Evalúa si el canal funciona diferente según la edad
# contact * perfil_deuda = Evalúa si el canal funciona diferente si el cliente tiene deudas
# tramo_balance = Evalúa si el saldo del cliente afecta la efectividad del canal de contacto

formula = """
deposit ~ C(contact) * age 
         + C(contact) * C(perfil_deuda) 
         + C(tramo_balance) 
         + C(poutcome) 
         + campaign
"""

modelo_definitivo = smf.logit(formula, data=df_marketing).fit()

# Ver el resumen estadístico completo
print(modelo_definitivo.summary())

Optimization terminated successfully.
         Current function value: 0.594888
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:                deposit   No. Observations:                10987
Model:                          Logit   Df Residuals:                    10971
Method:                           MLE   Df Model:                           15
Date:                Thu, 04 Jun 2026   Pseudo R-squ.:                  0.1409
Time:                        17:29:27   Log-Likelihood:                -6536.0
converged:                       True   LL-Null:                       -7608.0
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                                   coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------------------------
Intercept                               

In [36]:
# Extraer los Odds Ratios y sus P-valores para medir la significancia estadística
resultados_finales = pd.DataFrame({
    'Odds Ratio (OR)': np.exp(modelo_definitivo.params),
    'P-valor': modelo_definitivo.pvalues
})
print("\n--- RESULTADOS FINALES COMBINADOS (CAMINO 1 + 2) ---")
print(resultados_finales.round(4))


--- RESULTADOS FINALES COMBINADOS (CAMINO 1 + 2) ---
                                              Odds Ratio (OR)  P-valor
Intercept                                              1.8149   0.0000
C(contact)[T.telephone]                                0.3014   0.0001
C(contact)[T.unknown]                                  0.4188   0.0017
C(perfil_deuda)[T.1]                                   0.4634   0.0000
C(tramo_balance)[T.2. Saldo Bajo]                      0.9726   0.7441
C(tramo_balance)[T.3. Saldo Medio]                     1.3871   0.0000
C(tramo_balance)[T.4. Saldo Alto]                      1.8288   0.0000
C(poutcome)[T.no_campaign]                             0.8747   0.0420
C(poutcome)[T.other]                                   1.2949   0.0173
C(poutcome)[T.success]                                 8.2857   0.0000
C(contact)[T.telephone]:C(perfil_deuda)[T.1]           0.9306   0.6902
C(contact)[T.unknown]:C(perfil_deuda)[T.1]             1.8510   0.0000
age                    

Este modelo incluye las interacciones significativas (contact * age, contact * perfil_deuda) y el tramo_balance como control

___

Añadir :
probabilidad media predicha por segmentos x canal. 
- edad, (((¿clusters edad + trabajo + educación??- equipo perfil cliente. )))
- con deuda vs sin deuda ((ya tienen productos)) -- ?? 
- balance - ? - equipo finanzas 

añadir:

¿Qué porcentaje de la muestra tiene poutcome = success? Si es muy pequeño, el OR está inflado por pocos casos?

Distribución de poutcome por canal de contacto — puede haber confusión entre variables

Considerarlo como criterio de segmentación prioritaria para futuras campañas

añadir:

Gráfico de Odds Ratios con intervalos de confianza (forest plot)
Curva ROC
Heatmap de probabilidad predicha: canal × tramo de edad (o tramo de balance)

In [38]:
# Distribución de poutcome sobre el total de la muestra
poutcome_dist = df_marketing["poutcome"].value_counts(normalize=True).mul(100).round(2)
print("--- Distribución de poutcome (%) ---")
print(poutcome_dist.to_string())

--- Distribución de poutcome (%) ---
no_campaign    74.42
failure        11.03
success         9.73
other           4.82


In [39]:
# Frecuencia relativa de poutcome dentro de cada canal de contacto (por columna)
tabla_poutcome_canal = pd.crosstab(
    df_marketing["poutcome"],
    df_marketing["contact"],
    normalize="columns"
).mul(100).round(2)

print("--- poutcome por canal de contacto (%) ---")
print(tabla_poutcome_canal.to_string())

--- poutcome por canal de contacto (%) ---
contact      cellular  telephone  unknown
poutcome                                 
failure         14.23      10.21     0.26
no_campaign     67.42      72.64    99.17
other            6.02       6.02     0.31
success         12.34      11.13     0.26


poutcome = success representa el 9.73% de la muestra.
El OR ~8.3 está respaldado por volumen razonable (~1 de cada 10 clientes). No es un artefacto de pocos casos, el efecto es real y estable.

hallazgo:

no_campaign tiene 99.17 unknown --> estos son los nuevos clientes. sin campaña, , es un segmento sin historial.

confirmamos: 
cellular tiene más success (12.34%) -- ¿CATEGORÍA BASE? 
cellular tiene 14.23 de failure. 

telephone 11.13% exito, 
y menor el fail 10.21 %

____
Que cellular tenga más success no significa que el canal cause el éxito: puede ser que los clientes contactados por móvil sean un perfil diferente. Para aislar el efecto es necesario controlar por perfil de cliente

Filtrar variable "unknown" de contacto:

sin historial de campaña, sin poutcome real. Mezclarlo con cellular y telephone introduce ruido en los coeficientes del canal.

___
CLUSTERS sociodemográficos (equipo perfil de cliente) (?)


Perfiles con mayor propensión (ranking por tasa de conversión)

| Ranking | Perfil demográfico | Tasa de conversión |
|---|---|---|
| 1 | Jubilados · casados · ed. secundaria · ~61 años | 58.81% |
| 2 | Management · solteros · ed. terciaria · ~34 años | 56.10% |
| 3 | Técnicos · solteros · ed. secundaria · ~30 años | 50.43% |
| 4 | Blue-collar · casados · ed. secundaria · ~44 años | 40.24% |

El modelo tiene deposit, por lo cual NO es recomendable usar. Haremos: 

### tasa de conversión por categoría de perfil de cliente
Para cada variable (job, education, marital), calcular el porcentaje de deposit = 1 por categoría y ver cuánto varía entre el valor más alto y el más bajo. La variable con mayor rango de variación es la que más discrimina.

In [ ]:
## Cantidad de productos por cliente ##

product_cols = ["housing", "loan", "deposit"]

df_marketing["n_productos"] = (
    df_marketing[product_cols].sum(axis=1)
)

print("Clientes por cantidad de productos:")
print(df_marketing["n_productos"].value_counts().sort_index())

Clientes por cantidad de productos:
0    2056
1    6224
2    2442
3     265
Name: n_productos, dtype: int64


___
# JOB

In [42]:
# Tasa de conversión por categoría de job (rango = diferencia entre max y min)
job_conv = (df_marketing.groupby("job")["deposit"]
            .mean().mul(100).round(2)
            .sort_values(ascending=False))
print(f"--- Conversión por job (%) | Rango: {job_conv.max() - job_conv.min():.2f}pp ---")
print(job_conv.to_string())

--- Conversión por job (%) | Rango: 37.76pp ---
job
student          74.93
retired          67.01
unemployed       58.38
management       51.32
unknown          48.57
admin.           47.88
self-employed    46.98
technician       46.85
housemaid        40.82
services         40.73
entrepreneur     38.44
blue-collar      37.17


In [40]:
# Tasa de conversión por categoría de education (rango = diferencia entre max y min)
edu_conv = (df_marketing.groupby("education")["deposit"]
            .mean().mul(100).round(2)
            .sort_values(ascending=False))
print(f"--- Conversión por education (%) | Rango: {edu_conv.max() - edu_conv.min():.2f}pp ---")
print(edu_conv.to_string())

--- Conversión por education (%) | Rango: 14.61pp ---
education
tertiary     54.86
unknown      51.75
secondary    45.43
primary      40.25


In [41]:
# Tasa de conversión por categoría de marital (rango = diferencia entre max y min)
marital_conv = (df_marketing.groupby("marital")["deposit"]
                .mean().mul(100).round(2)
                .sort_values(ascending=False))
print(f"--- Conversión por marital (%) | Rango: {marital_conv.max() - marital_conv.min():.2f}pp ---")
print(marital_conv.to_string())

--- Conversión por marital (%) | Rango: 10.91pp ---
marital
single      55.07
divorced    48.74
married     44.16


confirmamos que job es la que más influye. 

education hay muchos unknown. estado civil son valores muy similares. 


____

### MODELO CON JOB x canal

In [44]:
# Excluir contactos sin canal identificado
df_model = df_marketing[df_marketing["contact"] != "unknown"].copy()

In [47]:
def extraer_resultados(modelo, etiqueta):
    """Extrae Odds Ratios y p-valores de un modelo logit."""
    resultados = pd.DataFrame({
        'Odds Ratio (OR)': np.exp(modelo.params),
        'P-valor': modelo.pvalues
    })
    print(f"\n--- {etiqueta} ---")
    print(resultados.round(4))
    return resultados

In [48]:
# Modelo 2 + job como control sociodemográfico
# Referencia: cellular, failure, sin deuda, saldo negativo
formula = """ 
deposit ~ C(contact) * age
         + C(contact) * C(perfil_deuda)
         + C(tramo_balance)
         + C(poutcome)
         + campaign
         + C(job)
"""
modelo_job = smf.logit(formula, data=df_model).fit()
resultados_job = extraer_resultados(modelo_job, "MODELO DEFINITIVO + JOB") ## extraer_resultados no está creado

Optimization terminated successfully.
         Current function value: 0.605212
         Iterations 6

--- MODELO DEFINITIVO + JOB ---
                                              Odds Ratio (OR)  P-valor
Intercept                                              2.1334   0.0000
C(contact)[T.telephone]                                0.3396   0.0007
C(perfil_deuda)[T.1]                                   0.5029   0.0000
C(tramo_balance)[T.2. Saldo Bajo]                      0.9405   0.5178
C(tramo_balance)[T.3. Saldo Medio]                     1.3692   0.0000
C(tramo_balance)[T.4. Saldo Alto]                      1.8847   0.0000
C(poutcome)[T.no_campaign]                             0.9021   0.1222
C(poutcome)[T.other]                                   1.2639   0.0333
C(poutcome)[T.success]                                 8.4043   0.0000
C(job)[T.blue-collar]                                  0.7943   0.0108
C(job)[T.entrepreneur]                                 0.7718   0.0905
C(job)[T.hous

In [49]:
# Probabilidad media de conversión predicha por perfil de cliente y canal
df_model["prob_predicha"] = modelo_job.predict(df_model)

prob_perfil = (df_model.groupby(["job", "contact"])["prob_predicha"]
               .mean().mul(100).round(2)
               .unstack("contact")
               .sort_values("cellular", ascending=False))

print("--- Probabilidad media predicha (%) por job x canal ---")
print(prob_perfil.to_string())

--- Probabilidad media predicha (%) por job x canal ---
contact        cellular  telephone
job                               
student           79.23      62.51
retired           70.70      76.58
unemployed        65.62      49.82
management        56.56      46.02
admin.            55.27      44.86
unknown           54.43      48.45
self-employed     53.94      42.17
technician        52.82      44.57
services          48.75      39.41
housemaid         48.32      38.38
blue-collar       45.76      36.20
entrepreneur      43.97      45.31


Patrón general: cellular supera a telephone en 10 de 12 perfiles. El canal móvil es sistemáticamente más efectivo.
Los perfiles de mayor edad (retired) responden mejor al teléfono fijo. 

____

### Recomendación para negocio
Priorizar cellular para todos los perfiles excepto retired, donde telephone tiene ventaja. Segmento student es el de mayor potencial diferencial por canal 

### objetivo y supuestos verificados

Regresión logística con C(job) como control. El objetivo era aislar el efecto del canal eliminando el sesgo de composición de perfiles. La alternativa descriptiva (tasa bruta por canal) no separaba si la diferencia venía del canal o del tipo de cliente contactado. El modelo logístico permite mantener fijos el resto de factores y extraer el efecto neto del canal.


¿Supuestos verificados?

Independencia de observaciones: asumida por diseño del dataset.
Ausencia de multicolinealidad severa: job y age están correlacionados (retirados = mayor edad) pero no perfectamente. La interacción contact × age ya estaba en el modelo y los coeficientes son estables.

___
# visualizaciones

Forest plot:
OR con IC 95% del modelo_job. Identificar qué predictores son significativos y con qué incertidumbre.

In [53]:
import plotly.io as pio
import plotly.express as px
import plotly.graph_objects as go

# DarkMode Plotly
pio.templates.default = "plotly_dark"
from plotly.subplots import make_subplots # permite visualizar varios gráficos

pio.templates["custom"] = pio.templates["plotly_dark"]
pio.templates["custom"].layout.paper_bgcolor = "#050a30"
pio.templates["custom"].layout.plot_bgcolor  = "#050a30"
pio.templates.default = "custom"

In [54]:
# Extraer OR, intervalo de confianza y significancia del modelo
ic = modelo_job.conf_int()
forest_df = pd.DataFrame({
    'OR':    np.exp(modelo_job.params),
    'IC_low': np.exp(ic[0]),
    'IC_high': np.exp(ic[1]),
    'pval':  modelo_job.pvalues
}).drop("Intercept").sort_values("OR", ascending=True)

forest_df["significativo"] = forest_df["pval"] < 0.05

In [ ]:
# Forest plot interactivo: OR con IC 95% — significativo en azul, no significativo en gris
fig = go.Figure()

for var, row in forest_df.iterrows():
    color = "#2E75B6" if row["significativo"] else "#888888"
    
    # Línea IC 95%
    fig.add_trace(go.Scatter(
        x=[row["IC_low"], row["IC_high"]],
        y=[var, var],
        mode="lines",
        line=dict(color=color, width=1.5),
        showlegend=False,
        hoverinfo="skip"
    ))
    
    # Punto OR
    fig.add_trace(go.Scatter(
        x=[row["OR"]],
        y=[var],
        mode="markers",
        marker=dict(color=color, size=7),
        showlegend=False,
        hovertemplate=f"<b>{var}</b><br>OR: {row['OR']:.3f}<br>p-valor: {row['pval']:.4f}<extra></extra>"
    ))

# Línea de referencia OR = 1
fig.add_vline(x=1, line_dash="dash", line_color="orange", line_width=0.8)

fig.update_layout(
    title="Forest Plot _ Modelo base + job. <br>Significativo en azul, no significativo en gris. Línea de referencia OR = 1",
    xaxis_title="Odds Ratio (IC 95%)",
    yaxis_title=None,
    height=len(forest_df) * 28,
    margin=dict(l=220, r=40, t=60, b=40)
)

fig.show()

In [66]:
# Verificar estructura de prob_perfil
print(prob_perfil.head())
print(prob_perfil.columns.tolist())

contact     cellular  telephone
job                            
student        79.23      62.51
retired        70.70      76.58
unemployed     65.62      49.82
management     56.56      46.02
admin.         55.27      44.86
['cellular', 'telephone']


___
### Ventaja porcentual JOB x CANAL

In [72]:
# Diferencia cellular vs telephone por perfil de cliente
diff_df = (prob_perfil["cellular"] - prob_perfil["telephone"]).sort_values(ascending=False)

fig = go.Figure(go.Heatmap(
    x=["cellular vs telephone"],
    y=diff_df.index.tolist(),
    z=[[v] for v in diff_df.values],
    colorscale=[[0, "#C00000"], [0.5, "#050a30"], [1, "#2E75B6"]],
    zmid=0,
    text=[[f"{v:+.2f}%"] for v in diff_df.values],
    texttemplate="%{text}",
    showscale=False
))

fig.update_layout(
    title="Ventaja por canal",
    height=len(diff_df) * 55,
    margin=dict(l=160, r=160, t=60, b=40),
    yaxis=dict(autorange="reversed")
)

fig.show()

____

### Probabilidad % job y poutcome

In [79]:
# Probabilidad media predicha por job x poutcome
heat_df = (df_model.groupby(["job", "poutcome"])["prob_predicha"]
           .mean().mul(100).round(2)
           .unstack("poutcome"))

fig = go.Figure(go.Heatmap(
    x=heat_df.columns.tolist(),
    y=heat_df.index.tolist(),
    z=heat_df.values,
    colorscale=[[0, "darkblue"], [1, "seagreen"]],
    text=heat_df.values.round(1),
    texttemplate="%{text}%",
    hovertemplate="<b>%{y} · %{x}</b><br>Prob: %{text}%<extra></extra>",
    showscale=False
))

fig.update_layout(
    title="Probabilidad media predicha (%) — job × poutcome",
    height=len(heat_df) * 55,
    margin=dict(l=160, r=160, t=60, b=40)
)

fig.show()

____

### Probabilidad media predicha por job x poutcome x contact

In [80]:
# Probabilidad media predicha por job x poutcome x contact
scatter_df = (df_model.groupby(["job", "poutcome", "contact"])["prob_predicha"]
              .mean().mul(100).round(2)
              .reset_index())

In [81]:
# Pivotar para tener cellular y telephone como columnas
scatter_pivot = scatter_df.pivot_table(
    index=["job", "poutcome"],
    columns="contact",
    values="prob_predicha"
).reset_index()

scatter_pivot.columns.name = None
print(scatter_pivot.head())

           job     poutcome  cellular  telephone
0       admin.      failure     50.92      44.38
1       admin.  no_campaign     48.80      40.21
2       admin.        other     57.09      41.96
3       admin.      success     90.96      87.25
4  blue-collar      failure     42.59      36.51


In [87]:
# Scatter: cellular vs telephone por job x poutcome
# Puntos sobre la diagonal = cellular gana, debajo = telephone gana

# Filtrar solo success y failure
scatter_filtered = scatter_pivot[scatter_pivot["poutcome"].isin(["success", "failure"])]

fig = px.scatter(
    scatter_filtered,
    x="telephone",
    y="cellular",
    color="poutcome",
    text="job",
    title="Cellular vs Telephone — probabilidad predicha por job × poutcome",
    labels={"telephone": "Telephone (%)", "cellular": "Cellular (%)"},
    hover_data=["job", "poutcome"]
)

# Línea diagonal de referencia y = x
fig.add_shape(type="line",
    x0=0, y0=0, x1=100, y1=100,
    line=dict(color="white", width=1, dash="dash")
)

fig.update_traces(textposition="bottom center", textfont_size=10)
fig.update_layout(height=600)
fig.show()

Este gráfico nos permite ver en detalle canal, job y poutcome